# Step 1: Data Modeling
1st script of "Integrating LLMs with Process Mining to Enhance Internal Audit Insights"

Purpose:<br>
Transforms curated warehouse tasks into a fit‑for‑mining event log.

In [0]:
# import and options
import re
import numpy as np
import datetime

# pyspark
from pyspark.sql.functions import (col, when, trim, concat, concat_ws, abs, substring, lit, year, lower, isnull, length, date_add, add_months, date_trunc, month, last_day, countDistinct, sum as spark_sum, min as spark_min, max as spark_max, first, last, count, round as spark_round, datediff, create_map, to_timestamp, expr, udf, lag, dayofmonth, year, month, split, lpad, left, right, timestamp_diff, mean as spark_mean, isnan)
from pyspark.sql.types import FloatType, TimestampType, IntegerType, StringType

## Requirements/Parameters
Path for import and export table in Unity Catalog

In [0]:
# TODO IMPORT: path to event list in Unity Catalog
# Format: "<unity_catalog>.<schema>.<table>"
delta_in_event_list = "<unity_catalog>.<schema>.<table>"

# TODO EXPORT: path to table for storage of created Process Mining events in Unity Catalog
# Format: "<unity_catalog>.<schema>.<table>"
delta_out_pm_events = "<unity_catalog>.<schema>.<table>"

## Load list of events

In [0]:

df_events = (
    spark
    .table(delta_in_event_list)
    .withColumn("ibl_obl_id", concat(col("ibl_trace_id"), lit("-"), col("obl_trace_id")))
    )
cols_events = df_events.columns

lines_count = df_events.count()
cases_count = df_events.select("ibl_obl_id").distinct().count()               
process_types_count = df_events.select("process_id-category-type").distinct().count()
locations_count = df_events.select("storage_type_id-text").distinct().count()

#df_events.printSchema()

print(f"Total {lines_count:,} rows with {cases_count:,} unique IBL-OBL cases.")
print(f"Number of process types: {process_types_count} & Number of locations: {locations_count}")

# Data Evaluation: prepare dataframes

In [0]:
df_ibl_obl_structure = (df_events
                        # Index: Combination of IBL_ & OBL_trace_id
                        .groupBy("ibl_trace_id", "obl_trace_id")
                        # pivot by LGTYP
                        .pivot("lgtyp")
                        .agg(countDistinct("event_id").alias("event_count"))
                        .fillna(0)
                        # check if combination has INDC/INGR/GR
                        .withColumn("gr_location_count", col("INDC") + col("INGR"))
                        .withColumn(
                            "process_stage", 
                            when(
                                col("gr_location_count") > 0, 
                                "1st stage")
                            .otherwise("2nd stage")
                            )
                        .orderBy(["ibl_trace_id", "obl_trace_id"], ascending=True)
                        .cache()
                        )

count_total = df_ibl_obl_structure.count()
count_1st = df_ibl_obl_structure.where(col("process_stage") == "1st stage").count()
count_2nd = df_ibl_obl_structure.where(col("process_stage") == "2nd stage").count()
print(f"Total rows: {count_total:,}; t/o 1st stage: {count_1st:,}, 2nd stage: {count_2nd:,}")


In [0]:
display(df_ibl_obl_structure.orderBy("gr_location_count", ascending=False))

## Check IBL-structure
On IBL-level:
- has 2nd stage?
- count of OBLs in 1st stage

In [0]:
# has 2nd stage?
# count OBLs in stages?
df_ibl_structure = (
    df_ibl_obl_structure
    .groupBy("ibl_trace_id")
    .pivot("process_stage")
    .agg(countDistinct("obl_trace_id").alias("obl_count"))
    .fillna(0)
    .orderBy("1st stage", ascending=False)
    .withColumn("ibl_has_2nd_stage", 
                when(col("2nd stage") > 0, lit(True))
                .otherwise(lit(False))
                )
    )

print(f"Rows {df_ibl_structure.count():,}")
display(df_ibl_structure)

In [0]:
display(df_ibl_structure.groupBy("ibl_has_2nd_stage").count())

## IBL/OBL-structures

In [0]:
df_ibl_obl = (
  df_ibl_obl_structure
  .select("ibl_trace_id", "obl_trace_id", "process_stage")
  .join(
    df_ibl_structure
    .select("ibl_trace_id", "ibl_has_2nd_stage"), 
    on="ibl_trace_id", 
    how="inner")
  .cache()
  )

ibl_obl_count = df_ibl_obl.count()
ibl_count = df_ibl_obl.select("ibl_trace_id").distinct().count()
obl_count = df_ibl_obl.select("obl_trace_id").distinct().count()

print(f"Total {ibl_obl_count:,} rows, {ibl_count:,} IBLs, {obl_count:,} OBLs")

## Join IBL/OBL-structures with event list

### 1st stage

In [0]:
# IBL/OBL of 1st stage
df_ibl_obl_1ststage = (
  df_ibl_obl
  .where(col("process_stage") == "1st stage")
  )

df_ibl_obl_1ststage.count()

In [0]:
# create event list of 1st stage
# rename for 1st stage
cols_1ststage_renamed = ["1ststage_" + c for c in cols_events]
dict_1ststage = dict(zip(cols_events, cols_1ststage_renamed))

df_events_1ststage = (
    df_ibl_obl_1ststage
    .select("ibl_trace_id", "obl_trace_id")
    .join(
        df_events,
        on=["ibl_trace_id", "obl_trace_id"],
        how="inner"
        )
    .withColumnsRenamed(dict_1ststage)
    .cache()
    )

df_events_1ststage.count()

### 2nd stage

In [0]:
# IBL/OBL of 2nd stage
df_ibl_obl_2ndstage = (
  df_ibl_obl
  .where(col("process_stage") == "2nd stage")
  )

df_ibl_obl_2ndstage.count()

In [0]:
# create event list of 2nd stage
# rename for 2nd stage
cols_2ndstage_renamed = ["2ndstage_" + c for c in cols_events]
dict_2ndstage = dict(zip(cols_events, cols_2ndstage_renamed))

df_events_2ndstage = (
    df_ibl_obl_2ndstage
    .select("ibl_trace_id", "obl_trace_id")
    .join(
        df_events,
        on=["ibl_trace_id", "obl_trace_id"],
        how="inner"
        )
    .withColumnsRenamed(dict_2ndstage)
    .cache()
    )

df_events_2ndstage.count()

# Data Modeling: Prepare final list of events

## Part 1: 1st stage events (when 2nd stage missing)
Filter these IBL/OBL combinations to extract the 1st stage events >> no adaption

In [0]:
df_ibl_obl.where(col("ibl_has_2nd_stage") == False).count()

In [0]:
df_p1_events = (
    df_ibl_obl
    .where(col("ibl_has_2nd_stage") == False)
    .select("ibl_trace_id", "obl_trace_id")
    .join(
      df_events_1ststage,
      on=[
        col("ibl_trace_id") == col("1ststage_ibl_trace_id"), 
        col("obl_trace_id") == col("1ststage_obl_trace_id")
        ],
      how="inner",
      )
    .drop(*["ibl_trace_id", "obl_trace_id"])
    .cache()
    )

part1_events = df_p1_events.count()
part1_ibl = df_p1_events.select("1ststage_ibl_trace_id").distinct().count()
part1_ibl_obl = (
  df_p1_events
  .withColumn("1ststage_ibl_obl_id", concat(col("1ststage_ibl_trace_id"), lit("-"), col("1ststage_obl_trace_id")))
  .select("1ststage_ibl_obl_id")
  .distinct()
  .count()
  )

print(f"Part 1 consists of {part1_events:,} events, {part1_ibl:,} IBLs, and {part1_ibl_obl:,} IBL/OBL-combinations")

## Part 2: all 2nd stage events
Take 2nd stage events in total >> no adaption

In [0]:
df_p2_events = df_events_2ndstage.cache()

df_p2_events.count()

## Part 3: Create 1st stage events (derrived from 2nd stage)

### 3a) single 1st stage OBL per IBL
For IBLs with a single 1st stage OBL: the mapping of 1st stage to 2nd stage is straight forward

In [0]:
# count IBLs with 2nd stage and single/no 1st stage OBL
df_p3a_ibl = (
    df_ibl_structure
    .where(
        (col("1st stage") == 1)
        & (col("ibl_has_2nd_stage") == True)
        )
    )

df_p3a_ibl.count()

In [0]:
# Filter 2nd stage events by IBLs
# return OBLs and first event to get quantity for 2nd stage
df_p3a_2ndstage_1stevents = (
    df_p3a_ibl
    .select("ibl_trace_id")
    .join(
        df_events_2ndstage,
        on=[col("ibl_trace_id") == col("2ndstage_ibl_trace_id")],
        how="inner"
    )
    .orderBy("2ndstage_timestamp", ascending=True)
    .groupBy("ibl_trace_id", "2ndstage_ibl_trace_id", "2ndstage_obl_trace_id")
    .agg(
        first("2ndstage_qty_nistm_actual").alias("2ndstage_qty_nistm_actual")
        )
    .cache()
    )

df_p3a_2ndstage_1stevents.count()

In [0]:
df_p3a_events = (df_p3a_2ndstage_1stevents
    .join(
        df_events_1ststage,
        on=[col("ibl_trace_id") == col("1ststage_ibl_trace_id")],
        how="inner"
    )
    # overwrite OBL and QTY
    .withColumn("1ststage_obl_trace_id", col("2ndstage_obl_trace_id"))
    .withColumn("1ststage_qty_nistm_actual", col("2ndstage_qty_nistm_actual"))
    .drop("ibl_trace_id", "2ndstage_ibl_trace_id", "2ndstage_obl_trace_id", "2ndstage_qty_nistm_actual")
)

df_p3a_events.count()

### 3b) multiple 1st stage OBLs per IBL
For IBLs with a multiple 1st stage OBLs: <br>
Mapping and duplicating 1st stage to 2nd stage requires a step-wise approach:
- identify IBL matching criteria
- IBL filter 2nd stage events to find FIRST events with LGTYP, LOCATION, and QTY
- IBL filter 1st stage events to find LAST events with LGTYP, and LOCATION

In [0]:
# count IBLs with 2nd stage and mutiple 1st stage OBLa
df_p3b_ibl = (
    df_ibl_structure
    .where(
        (col("1st stage") > 1)
        & (col("ibl_has_2nd_stage") == True)
        )
    )

df_p3b_ibl.count()

In [0]:
# Filter 2nd stage events by IBLs
# return OBLs and first event to get location and quantity for 2nd stage
df_p3b_2ndstage_1stevents = (
    df_p3b_ibl
    .select("ibl_trace_id")
    .join(
        df_events_2ndstage,
        on=[col("ibl_trace_id") == col("2ndstage_ibl_trace_id")],
        how="inner"
    )
    .orderBy("2ndstage_timestamp", ascending=True)
    .groupBy("ibl_trace_id", "2ndstage_ibl_trace_id", "2ndstage_obl_trace_id")
    .agg(
        first("2ndstage_timestamp").alias("2ndstage_timestamp"),
        first("2ndstage_lgtyp").alias("2ndstage_lgtyp"),
        first("2ndstage_location").alias("2ndstage_location"),
        first("2ndstage_qty_nistm_actual").alias("2ndstage_qty_nistm_actual")
        )
    .drop("ibl_trace_id")
    .cache()
    )

df_p3b_2ndstage_1stevents.count()

In [0]:
# Filter 1st stage events by IBLs
# return OBLs and LAST event to get location for 1st stage
df_p3b_1ststage_lastevent = (
    df_p3b_ibl
    .select("ibl_trace_id")
    .join(
        df_events_1ststage,
        on=[col("ibl_trace_id") == col("1ststage_ibl_trace_id")],
        how="inner"
    )
    .orderBy("1ststage_timestamp", ascending=True)
    .groupBy("ibl_trace_id", "1ststage_ibl_trace_id", "1ststage_obl_trace_id")
    .agg(
        last("1ststage_timestamp").alias("1ststage_timestamp"),
        last("1ststage_lgtyp").alias("1ststage_lgtyp"),
        last("1ststage_location").alias("1ststage_location"),
        )
    .drop("ibl_trace_id")
    .cache()
    )

df_p3b_1ststage_lastevent.count()

In [0]:
df_p3b_1ststage_obl_to_2ndstage = (
    df_p3b_2ndstage_1stevents
    .join(
        df_p3b_1ststage_lastevent, 
        on=[
            col("2ndstage_ibl_trace_id") == col("1ststage_ibl_trace_id"),
            col("2ndstage_timestamp") >= col("1ststage_timestamp"),
            col("2ndstage_lgtyp") == col("1ststage_lgtyp"),
            col("2ndstage_location") == col("1ststage_location"),
            ], 
        how="left"
        )
    .cache()
    )

df_p3b_1ststage_obl_to_2ndstage.count()

In [0]:
# remove duplicates and take first matched OBL-event only
# result: IBL/OBLs to filter 1st stage events
df_p3b_1ststage_obl = (
    df_p3b_1ststage_obl_to_2ndstage
    # remove failed matches
    .where(col("1ststage_obl_trace_id").isNotNull())
    .orderBy("1ststage_timestamp", ascending=True)
    .groupBy("1ststage_ibl_trace_id", "2ndstage_obl_trace_id", "2ndstage_qty_nistm_actual")
    .agg(
        first("1ststage_obl_trace_id").alias("1ststage_obl_trace_id"),
        )
)
df_p3b_1ststage_obl.count()

In [0]:
df_p3b_events = (
    df_p3b_1ststage_obl
    .join(df_events_1ststage,
          on=["1ststage_ibl_trace_id", "1ststage_obl_trace_id"],
          how="inner"
          )
    # overwrite OBL and QTY
    .withColumn("1ststage_obl_trace_id", col("2ndstage_obl_trace_id"))
    .withColumn("1ststage_qty_nistm_actual", col("2ndstage_qty_nistm_actual"))
    .drop("2ndstage_ibl_trace_id", "2ndstage_obl_trace_id", "2ndstage_qty_nistm_actual")
    .cache()
)

df_p3b_events.count()

### 3c) no OBL in 1st stage
no need for action

### 3d) OBLs in 1st stage unmatched (in part 3b)
Check which IBL/OBLs of 1st stage are not matched in part 3b.

In [0]:
df_p3d_events = (
    df_events_1ststage
    # filter by IBL-scope of part 3b
    .join(
        df_p3b_ibl
        .select("ibl_trace_id"),
        on=[col("1ststage_ibl_trace_id") == col("ibl_trace_id")],
        how="inner")
    # compare with successful matches in part 3b
    .join(
        df_p3b_1ststage_obl_to_2ndstage
        .select("1ststage_ibl_trace_id", "1ststage_obl_trace_id")
        .where(col("1ststage_obl_trace_id").isNotNull()),
        on=["1ststage_ibl_trace_id", "1ststage_obl_trace_id"],
        how="left_anti"
        # left anti join: Returns all rows from the left table (df_events_1ststage) for which there is no match in the right table (successful matches)
    )
    .drop("ibl_trace_id")
    .cache()
    )

df_p3d_events.count()    

# Union and export

In [0]:
df_pm_events_modelled = (
    # 1st stage events
    df_p1_events
    .unionByName(df_p3a_events)
    .unionByName(df_p3b_events)
    .unionByName(df_p3d_events)
    .withColumnsRenamed(dict(zip(cols_1ststage_renamed, cols_events)))
    # 2nd stage events
    .unionByName(
        df_p2_events
        .withColumnsRenamed(dict(zip(cols_2ndstage_renamed, cols_events)))
        )
    .withColumn("ibl_obl_id", concat(col("ibl_trace_id"), lit("-"), col("obl_trace_id")))
    .cache()
    )

df_pm_events_modelled.count()

In [0]:
# export table
df_pm_events_modelled = df_pm_events_modelled.persist()         # or .cache()
lines_count = df_pm_events_modelled.count()                      # triggers materialization
df_pm_events_modelled.writeTo(delta_out_pm_events).createOrReplace()

ibl_count = df_pm_events_modelled.select("ibl_trace_id").distinct().count()
obl_count = df_pm_events_modelled.select("obl_trace_id").distinct().count()
ibl_obl_count = df_pm_events_modelled.select("ibl_obl_id").distinct().count()
print(f"Total {lines_count:,} rows with {ibl_obl_count:,} combinations of IBL/OBL")
print(f"Number of IBL_trace_IDs: {ibl_count:,} & Number of OBL_trace_IDs: {obl_count:,}")